# Flow outputs and polarity

Flow mass is a queryable edge output. Aggregate with an explicit `side=`
(`"source"` or `"dest"`), turn instantaneous rates into counts with
`.integrate_intervals()`, and compare the two sides of the same flow.

Ageing is the picture for polarity: at t = 0 nobody arrives in 0–4
and nobody leaves 10+. The interval integral of the infection rate is
drawn by age of the destination. The integral
bars are a known function, `t²` on `[0, 1]`, so you can see Simpson
close more of the gap than trapezoid. The last bars are Kleene on an
exit: death has no destination, so `Dest(Everything())` keeps nothing.

The last four sections check output algebra: an age-specific flow divided
by the people in that age, a cumulative that starts on a save time, the
summer2 midpoint convention, and one `FlowMass` that sums two flows.


In [ ]:
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

import numpy as np

from summer4 import (
    Compartments,
    Dest,
    Everything,
    ExitFlow,
    FlowMass,
    FlowModel,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Source,
    TraitChain,
    TransitionFlow,
)

state = Property("state", ("S", "I", "R"))
age = Property("age", ("0-4", "5-9", "10+"))
pmap = PropertyMap.from_property(state).stratify(age)

model = FlowModel(pmap)
model.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        0.3,
    )
)
model.add_flow(
    TransitionFlow(
        "ageing",
        age.present(),
        age.present(),
        0.2,
        pairing=TraitChain(age, (("0-4", "5-9"), ("5-9", "10+"))),
    )
)
model.add_flow(ExitFlow("death", state["I"], 0.05))
cm = model.compile()

y0 = PropertyData.wrap(pmap, np.ones(pmap.size))
y0 = y0.at[state["S"] & age["0-4"]].set(500.0)
y0 = y0.at[state["I"] & age["0-4"]].set(10.0)

plan = SavePlan(
    requests={
        "infection": SaveRequest(FlowMass(flow="infection")),
        "ageing": SaveRequest(FlowMass(flow="ageing")),
        "death": SaveRequest(FlowMass(flow="death")),
    }
)
res = cm.run({}, y0, t0=0.0, steps=28, dt=1.0, save=plan)
assert res["infection"].dims == ("time", "edge")


## Source vs dest on the same flow

Ageing moves people from one band to the next. Grouping by the **source** age
counts who left each band; grouping by the **dest** age counts who arrived.
Neither is a default — `side=` is required. At the first save, the
oldest band should have left nobody (there is no older band to enter)
and the youngest band should have received nobody.


In [ ]:
by_src = res["ageing"].sum_over(age, side="source").total()
by_dest = res["ageing"].sum_over(age, side="dest").total()
src_frame = res["ageing"].sum_over(age, side="source").to_frame()
dest_frame = res["ageing"].sum_over(age, side="dest").to_frame()
assert list(src_frame.columns)[1:] == ["age=0-4", "age=5-9", "age=10+"]
assert list(dest_frame.columns)[1:] == ["age=0-4", "age=5-9", "age=10+"]
# Nobody ages *out* of 10+; nobody ages *into* 0-4.
assert float(np.asarray(by_src.values)[0]) > 0.0
src_at_0 = np.asarray(res["ageing"].sum_over(age, side="source").values.data)[0]
dest_at_0 = np.asarray(res["ageing"].sum_over(age, side="dest").values.data)[0]
assert float(src_at_0[2]) == 0.0
assert float(dest_at_0[0]) == 0.0
assert not np.allclose(src_at_0, dest_at_0)
print("source age masses at t0:", src_at_0)
print("dest age masses at t0:  ", dest_at_0)

bands = ["0-4", "5-9", "10+"]
pd.DataFrame(
    {"left this band": src_at_0, "arrived in this band": dest_at_0},
    index=bands,
).plot.bar(
    title="Ageing at t = 0: source age is not destination age",
    labels={"index": "age", "value": "people / day"},
)
res["ageing"].sum_over(age, side="source").to_pandas().plot(
    title="Ageing outflow by source age",
    labels={"index": "time", "value": "people / day"},
)
res["ageing"].sum_over(age, side="dest").to_pandas().plot(
    title="Ageing inflow by destination age",
    labels={"index": "time", "value": "people / day"},
)


## New infections per save interval, by age of arrival

Saved flow mass is an **instantaneous rate**. `.integrate_intervals()` integrates each
save interval (trapezoid by default) to a count; times become the interval
right edges. Each line is new infections arriving in that age band.


In [ ]:
weekly = (
    res["infection"]
    .integrate_intervals()
    .sum_over(age, side="dest")
    .to_frame()
)
assert "age=0-4" in weekly.columns
assert weekly.height == 28  # T-1 intervals from 29 save points (steps+1)
print(weekly.head())

res["infection"].integrate_intervals().sum_over(age, side="dest").to_pandas().plot(
    title="New infections per interval, by age of arrival",
    labels={"index": "time", "value": "people per interval"},
)


## Rate vs interval integral: the quadrature gap

Trapezoid over save intervals is second-order accurate; it is **not** the
solver's own accumulated flow. The difference is visible and quantified
below — save finer (or use `method="simpson"`) when you need the integral, not the rate.
The curve is `f(t) = t²`. The bars are its integral on `[0, 1]`, whose
true value is 1/3. Simpson should land closer than trapezoid.


In [ ]:
# Analytic stand-in: integrate f(t)=t^2 on [0, 1].
from summer4.results.output import Output
from summer4.time import TimeAxis

ts = np.linspace(0.0, 1.0, 5)
rate = Output(
    times=TimeAxis(values=ts, epoch=None, kind="explicit"),
    values=ts**2,
    dims=("time",),
)
analytic = 1.0 / 3.0
trap = float(np.asarray(rate.integrate(method="trapezoid").values))
simp = float(np.asarray(rate.integrate(method="simpson").values))
print(f"analytic={analytic:.6f}  trapezoid={trap:.6f}  err={abs(trap - analytic):.2e}")
print(f"analytic={analytic:.6f}  simpson={simp:.6f}    err={abs(simp - analytic):.2e}")
assert abs(simp - analytic) < abs(trap - analytic)

rate.to_pandas().plot(
    title="Rate whose integral we know: f(t) = t²",
    labels={"index": "t", "value": "f(t)"},
)
pd.Series(
    {"analytic": analytic, "trapezoid": trap, "simpson": simp}
).to_frame("integral").plot.bar(
    title="Simpson closes more of the gap to ∫t² dt = 1/3",
    labels={"index": "method", "value": "integral on [0, 1]"},
)


## Kleene-unknown on an exit flow

`Dest(Everything())` on a death flow selects nothing (the destination side is
absent). The same selector on a transition flow keeps every edge. The bars
should be 0 for death and the full infection edge count for infection.


In [ ]:
death_dest = res["death"].select(Dest(Everything()))
inf_dest = res["infection"].select(Dest(Everything()))
assert death_dest.values.pmap.size == 0
assert inf_dest.values.pmap.size == res["infection"].values.pmap.size

# Only edges that actually change age:
moved = res["ageing"].select(cm.edges("ageing").moves_mask(age))
assert moved.values.pmap.size == res["ageing"].values.pmap.size
print("exit Dest(Everything()) edges:", death_dest.values.pmap.size)
print("infection Dest(Everything()) edges:", inf_dest.values.pmap.size)

pd.Series(
    {
        "death Dest(Everything())": death_dest.values.pmap.size,
        "infection Dest(Everything())": inf_dest.values.pmap.size,
    }
).to_frame("edges").plot.bar(
    title="An exit has no destination, so Dest(Everything()) is empty",
    labels={"index": "selector", "value": "edges kept"},
)


## Per-capita infection by age

Divide the infection flow, grouped by the age it arrives in, by the number of
people in that age. The lines are new infections per person per day. At t = 0
the susceptible counts are known (500 in 0–4, one in each older band), so the
ratio is `0.3 * S / N` and can be checked against the figure.

In [ ]:
pop_plan = SavePlan(
    requests={
        "compartments": SaveRequest(Compartments()),
        "infection": SaveRequest(FlowMass(flow="infection")),
    }
)
pop_res = cm.run({}, y0, t0=0.0, steps=28, dt=1.0, save=pop_plan)
by_age = pop_res["infection"].sum_over(age, side="dest")
population = pop_res["compartments"].sum_over(age)
per_capita = by_age / population
assert per_capita.dims == ("time", "group")
assert isinstance(per_capita.values, PropertyData)

s0 = np.array([500.0, 1.0, 1.0])
n0 = np.array([511.0, 3.0, 3.0])
np.testing.assert_allclose(
    np.asarray(per_capita.values.data)[0],
    0.3 * s0 / n0,
    rtol=1e-5,
)
per_capita.to_pandas().plot(
    title="Infection per person, by age of arrival",
    labels={"index": "time", "value": "infections per person per day"},
)

## Cumulative deaths from day 14

`cumulative(start=14)` is zero on earlier save times and a running sum from
day 14, which is itself a save time. The curve should sit on zero until then
and start at that day's death rate, not at the sum of everything before it.

In [ ]:
deaths = res["death"].total()
from_day_14 = deaths.cumulative(start=14.0)
assert float(np.asarray(from_day_14.values)[13]) == 0.0
np.testing.assert_allclose(
    float(np.asarray(from_day_14.values)[14]),
    float(np.asarray(deaths.values)[14]),
    rtol=1e-5,
)
from_day_14.to_pandas().plot(
    title="Cumulative deaths, counting only from day 14",
    labels={"index": "time", "value": "people / day, summed"},
)

## Midpoint, the summer2 flow convention

Summer2's default flow output (`raw_results=False`) keeps the first sample and
replaces every later sample with the average of that sample and the one before
it. `midpoint()` does that, so a port can compare numbers. It is not the
solver's accumulated flow — the quadrature section above is the honest
integral, and this is the parity convention. The two lines should meet at
t = 0 and then the midpoint line should sit between neighbouring raw points.

In [ ]:
raw = np.asarray(deaths.values)
mid = deaths.midpoint()
assert float(np.asarray(mid.values)[0]) == float(raw[0])
np.testing.assert_allclose(
    np.asarray(mid.values)[1:],
    0.5 * (raw[1:] + raw[:-1]),
    rtol=1e-5,
)
pd.DataFrame(
    {"saved rate": raw, "midpoint": np.asarray(mid.values)},
    index=np.asarray(deaths.times.values),
).plot(
    title="Summer2 midpoint convention is not the raw death rate",
    labels={"index": "time", "value": "people / day"},
)

## One save that sums infection and death

`FlowMass` can name several flows. They are filtered the same way and added.
Here that filter is source age, so the bars are people leaving each age by
infection or by death. The combined save should match adding the two flows
after the fact, and at t = 0 that sum is `0.3 * S + 0.05 * I`.

In [ ]:
combined_plan = SavePlan(
    requests={
        "leaving": SaveRequest(
            FlowMass(flow=("infection", "death"), sum_over=(age, "source"))
        )
    }
)
combined = cm.run({}, y0, t0=0.0, steps=28, dt=1.0, save=combined_plan)
manual = res["infection"].sum_over(age, side="source") + res["death"].sum_over(
    age, side="source"
)
np.testing.assert_allclose(
    np.asarray(combined["leaving"].values.data),
    np.asarray(manual.values.data),
    rtol=1e-5,
)
s0 = np.array([500.0, 1.0, 1.0])
i0 = np.array([10.0, 1.0, 1.0])
np.testing.assert_allclose(
    np.asarray(combined["leaving"].values.data)[0],
    0.3 * s0 + 0.05 * i0,
    rtol=1e-5,
)
pd.Series(
    np.asarray(combined["leaving"].values.data)[0],
    index=["0-4", "5-9", "10+"],
).to_frame("people / day").plot.bar(
    title="Infection plus death at t = 0, by source age",
    labels={"index": "age", "value": "people / day"},
)